**Google Drive Mounting**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile
import os

zip_path = '/content/archive.zip'
extract_path = '/content/'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"'{zip_path}' extracted to '{extract_path}'")
else:
    print(f"Error: '{zip_path}' not found.")


'/content/archive.zip' extracted to '/content/'


**Imports**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator


**Dataset Loading**

In [ ]:
data_root = '/content/data'
train_dir = os.path.join(data_root, 'train')
val_dir = os.path.join(data_root, 'val')
test_dir = os.path.join(data_root, 'test')

**Defining Parameters**

In [ ]:
img_width, img_height = 32, 32 # resizing to 32x32
batch_size = 32
epochs_head = 20
epochs_fine = 30
lr_head = 1e-3
lr_fine = 1e-4
classes = ['awake', 'sleepy'] # clearing classes order

**Data Augmentation & Loading**



In [ ]:
train_gen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range = 0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.3],
)

val_test_gen = ImageDataGenerator(rescale=1.0/255)

train_ds = train_gen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    color_mode='rgb',
    class_mode='categorical',
    classes=classes,
    batch_size=batch_size,
    shuffle=True,
    seed=42
)

val_ds = val_test_gen.flow_from_directory(
    val_dir,
    target_size=(img_width, img_height),
    color_mode='rgb',
    class_mode='categorical',
    classes=classes,
    batch_size=batch_size,
    shuffle=False
)
test_ds = val_test_gen.flow_from_directory(
    test_dir,
    target_size=(img_width, img_height),
    color_mode='rgb',
    class_mode='categorical',
    classes=classes,
    batch_size=batch_size,
    shuffle=False
)
print('class index: ', train_ds.class_indices)

Found 50937 images belonging to 2 classes.
Found 16980 images belonging to 2 classes.
Found 16981 images belonging to 2 classes.
class index:  {'awake': 0, 'sleepy': 1}


**Choosing MobileNetV2 As The Base Model**

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(img_width, img_height, 3)
)
base_model.trainable = False

/tmp/ipykernel_5272/741209991.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


**Model Head Implementation**

In [ ]:
inputs = keras.Input(shape=(img_width, img_height, 3))
x = keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(classes), activation='softmax')(x)
model = keras.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 1, 1, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,427,330 (9.26 MB)

 Trainable params: 166,786 (651.51 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

# Phase 1: Head-Only Training

In [ ]:
model.compile(
    optimizer=Adam(lr_head),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

**Callbacks**

In [ ]:
checkpoint = ModelCheckpoint(
    'best_model_phase1.keras',
    monitor='val_accuracy',
    save_best_only=True
)
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=4,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
)

In [ ]:
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs_head,
    callbacks=[checkpoint, early_stopping, reduce_lr]
)

Epoch 1/20
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 109s 63ms/step - accuracy: 0.7202 - loss: 0.5384 - val_accuracy: 0.7870 - val_loss: 0.4494 - learning_rate: 0.0010
Epoch 2/20
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 84s 53ms/step - accuracy: 0.7448 - loss: 0.5064 - val_accuracy: 0.8048 - val_loss: 0.4313 - learning_rate: 0.0010
Epoch 3/20
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 84s 53ms/step - accuracy: 0.7551 - loss: 0.4962 - val_accuracy: 0.8166 - val_loss: 0.4187 - learning_rate: 0.0010
Epoch 4/20
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 83s 52ms/step - accuracy: 0.7583 - loss: 0.4905 - val_accuracy: 0.8265 - val_loss: 0.4093 - learning_rate: 0.0010
Epoch 5/20
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 143s 53ms/step - accuracy: 0.7639 - loss: 0.4849 - val_accuracy: 0.8232 - val_loss: 0.4097 - learning_rate: 0.0010
Epoch 6/20
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 85s 53ms/step - accuracy: 0.7644 - loss: 0.4818 - val_accuracy: 0.8309 - val_loss: 0.4089 - learning_rate: 0.0010
Epoch 7/20
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 82s 52ms/step - accu

# Phase 2: Fine-Tuning

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
  layer.trainable = False

In [ ]:
model.compile(
    optimizer=Adam(lr_fine),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# reseting callbacks
checkpoint = ModelCheckpoint(
    'best_model_phase2.keras',
    monitor='val_accuracy',
    save_best_only=True
)
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=4,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
)

In [ ]:
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs_fine,
    callbacks=[checkpoint, early_stopping, reduce_lr],
    initial_epoch=len(history1.epoch)
)

Epoch 21/30
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 85s 54ms/step - accuracy: 0.8521 - loss: 0.3325 - val_accuracy: 0.9077 - val_loss: 0.2351 - learning_rate: 1.2500e-05
Epoch 22/30
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 83s 52ms/step - accuracy: 0.8597 - loss: 0.3228 - val_accuracy: 0.9092 - val_loss: 0.2250 - learning_rate: 1.2500e-05
Epoch 23/30
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 86s 54ms/step - accuracy: 0.8613 - loss: 0.3190 - val_accuracy: 0.9206 - val_loss: 0.2079 - learning_rate: 1.2500e-05
Epoch 24/30
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 86s 54ms/step - accuracy: 0.8639 - loss: 0.3095 - val_accuracy: 0.9190 - val_loss: 0.2013 - learning_rate: 1.2500e-05
Epoch 25/30
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 84s 53ms/step - accuracy: 0.8705 - loss: 0.3024 - val_accuracy: 0.9162 - val_loss: 0.2001 - learning_rate: 1.2500e-05
Epoch 26/30
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 85s 54ms/step - accuracy: 0.8698 - loss: 0.3043 - val_accuracy: 0.9220 - val_loss: 0.1996 - learning_rate: 1.2500e-05
Epoch 27/30
1592/1592 ━━━━━━━━━━━━

**Evaluation**

In [ ]:
loss, accuracy = model.evaluate(test_ds)
print(f"Test accuracy: {round(accuracy * 100, 2)}%")
print(f"Test loss: {round(loss, 2)}")

531/531 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9319 - loss: 0.1772
Test accuracy: 93.19%
Test loss: 0.18


**Saving Keras Model**

In [ ]:
model.save('model.keras')

**Converting To TFLite**

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open('model.tflite', 'wb') as f:
  f.write(tflite_model)
print(len(tflite_model)/1024, 'KB')

Saved artifact at '/tmp/tmppvyn3zon'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='keras_tensor_329')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  140472427692880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140472431369232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140472431376720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140472427692112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140472431376528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140472431378064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140472431369808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140472431371344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140472431377488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140472431373648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140472431372